# Reproducing See et al. (2018) from CRCNS ac-3

**Paper.** See JZ, Atencio CA, Sohal VS, Schreiner CE (2018). Coordinated neuronal ensembles in
primary auditory cortical columns. *eLife* 7:e35587,
[doi:10.7554/eLife.35587](https://doi.org/10.7554/eLife.35587). The claims file
(`claims/see-2018.yaml`) was written from the version of record (PMC6017807) with its figure
legends, Materials and methods, and Supplementary file 1.

**Data set.** [CRCNS ac-3](https://doi.org/10.6080/K09021X1) (See, Atencio & Schreiner 2018),
read from the public bucket `crcnsarchive` under `ac-3/live/`, zip files opened in place.
CRCNS data sets are not versioned, so the version is recorded as `unversioned`, the only one
published; the paper cites the data set by its DOI and no version. The files read have these MD5
sums in the data set's `checksums.md5`: `Data.zip` 3140d6178de9c044932901795a4cc817,
`Stimulus.zip` ce43beade196882598cecddff5520d7b (the notebook prints the listing it read).
The data descriptor is version 0.6 (March 28, 2018).

**What the data set holds.** 16 recordings (`site1` to `site16`), 8 with the DMR (`rn1`) and 8
with the RN (`rn16`), 655 units, the 10-min stimuli and their triggers: the 32-channel columnar
recordings behind Figures 1 to 7, 10 and 11C to F. It does not hold the 50 repeats of the 5-s
segments (Figure 9 and its supplements), the 15 repeats of the 10-min RN (Figure 11A, B and the
'repeat' CRs of Figure 11E, F) or the 64-channel recordings with spontaneous and evoked epochs
(Figure 8).

| Claim | Tested here |
|---|---|
| fig2bd-pwc-measures-covary | tested |
| fig2eg-distance-weak | tested |
| fig4-example-nine-cnes | tested |
| fig4s2-interleaved-halves-stable | tested |
| fig4s3-train-test-83pct | tested |
| fig5a-mi-peaks-10-15ms | tested |
| fig5b-number-decreases-with-bin | tested |
| fig5c-size-increases-with-bin | tested |
| fig6c-spike-pwc-narrower-than-strf-pwc | tested |
| fig6j-sharpness-cne-pairs | tested |
| fig6-member-pwc-sharpness-delay | tested |
| fig7-totals, fig7a, fig7b, fig7-negative-weights, fig7c, fig7d, fig7e, fig7f | tested |
| fig8-spont-evoked-72pct | blocked: the 64-channel recordings are not in the data set |
| fig9c-cne-events-more-mi, fig9s1-cne-mi-above-random-groups, fig9s2-noise-correlations | blocked: the 5-s repeats are not in the data set |
| fig10b-ptd-higher, fig10c-strf-mi-higher, fig10s1-random-group-strfs | tested |
| fig11ab-real-vs-surrogate | blocked: the 15 repeats of the 10-min RN are not in the data set |
| fig11c-models-number, fig11d-models-size | tested |
| fig11ef-coincidence-ratios | tested on the three models; the 'repeat' CR needs the repeats |
| fig11s3b-pr-strf-similarity | tested |

Loading and measures are in `cne.py`; its docstring says where each parameter comes from.
Heavy steps run in four worker processes, one recording each. Every section has its own seed.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"
import itertools
import multiprocessing as mp
import resource
import time
import warnings
from concurrent.futures import ProcessPoolExecutor
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import cne

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()

print(f"data set {cne.DATASET}, version {cne.VERSION}, from {cne.BASE}")
print(urlopen(f"{cne.BASE}/checksums.md5").read().decode())
cne.SITES = cne.load_sites()
for s in cne.SITES:
    cne.prepare(s)
params = {n: cne.load_params(n) for n in ("rn1", "rn16")}
stim_info = {}
for name in ("rn1", "rn16"):
    raw = cne.load_stimulus(name)
    stim_info[name] = {"shape": raw.shape, "min": float(raw.min()), "max": float(raw.max()),
                       "mean": float(raw.mean())}
    cne.STIM[(name, cne.STRF_BIN_MS)] = cne.downsample(raw, cne.STRF_BIN_MS)
    cne.STIM[(name, 1)] = cne.downsample(raw, 1)
    del raw
IDX = list(range(len(cne.SITES)))
SITE_IDS = [s["site"] for s in cne.SITES]


def pmap(fn, *args):
    with ProcessPoolExecutor(4, mp_context=mp.get_context("fork")) as ex:
        return list(ex.map(fn, *args))


print(f"loaded {len(cne.SITES)} recordings and 2 stimuli in {time.time() - started:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

Each check of the descriptor against the values is stated under its output.

In [ ]:
rows = []
for s in cne.SITES:
    trig_ms = s["trigger"] / cne.FS * 1000
    sp = np.concatenate([u["spikes_ms"] for u in s["units"]])
    st = np.concatenate([u["stim_ms"] for u in s["units"]])
    rows.append({"site": s["site"], "file": s["file"], "stim": s["stim"],
                 "unit_stim": ",".join(sorted({u["stim"] for u in s["units"]})),
                 "units": len(s["units"]), "tip_depth_um": ",".join(str(int(v)) for v in sorted({u["tip_depth"] for u in s["units"]})),
                 "fs": ",".join(str(int(v)) for v in sorted({u["fs"] for u in s["units"]})),
                 "triggers": len(s["trigger"]),
                 "trig_interval_ms": f"{np.diff(trig_ms).min():.2f}-{np.diff(trig_ms).max():.2f}",
                 "first_trig_s": round(trig_ms[0] / 1000, 2),
                 "stim_s": round(cne.stim_duration_ms(s) / 1000, 1),
                 "spikes": len(sp), "spike_s": f"{sp.min() / 1000:.1f}-{sp.max() / 1000:.1f}",
                 "in_stimulus": round(float(np.isfinite(st).mean()), 3),
                 "depth_um": f"{min(u['depth'] for u in s['units']):.0f}-{max(u['depth'] for u in s['units']):.0f}",
                 "lateral_um": ",".join(str(int(v)) for v in sorted({u["x"] for u in s["units"]})),
                 "sorted": all(np.all(np.diff(u["spikes_ms"]) >= 0) for u in s["units"])})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
for s in cne.SITES:
    for k, u in enumerate(s["units"]):
        d = np.diff(u["spikes_ms"])
        if (d < 0).any():
            print(f"site {s['site']} unit {k}: {(d < 0).sum()} decreasing steps of {len(d)} (largest {-d.min():.2f} ms), "
                  f"{(d == 0).sum()} repeated times")
print("\nrecordings", len(info), "| units", info.units.sum(), "| by stimulus", info.groupby("stim").units.agg(["count", "sum"]).to_dict())
for name, p in params.items():
    print(f"{name}: NF {p['NF']}, NT {p['NT']}, taxis step {np.diff(p['taxis'])[0] * 1000:.2f} ms over "
          f"{p['taxis'][-1] * 1000:.0f} ms, f1 {p['f1']} Hz, f2 {p['f2']} Hz, MaxFM {p['MaxFM']} Hz, "
          f"MaxRD {p['MaxRD']} cyc/oct, FM shape {np.shape(p['FM'])}")
for name, v in stim_info.items():
    n_trig = sorted(set(info[info.stim == name].triggers))
    print(f"{name} stim_mat {v['shape']}, {v['shape'][1] / cne.NT:.1f} trigger periods (triggers in its files: {n_trig}); "
          f"range {v['min']:.1f} to {v['max']:.1f}, mean {v['mean']:.2f}")

The descriptor's counts and units hold. There are 16 files, 8 per stimulus, and the `stim` field
of every unit matches its file name. The 655 units match the paper's total. Every `fs` is
20 kHz. The triggers come every 333.3 ms (6666 or 6667 samples), which is the parameter files'
`taxis` of 667 columns of 0.5 ms. Each stimulus matrix has exactly 667 columns per trigger of
its files (1795 for rn1, 1799 for rn16), so it covers about 598 s, the "10-min" stimulus. The
envelope runs from -20 to 20 dB, the 40 dB modulation depth of the Methods, over 64 carriers
from 500 Hz to 40 kHz. Spike times are in ms, and sorted except in two units (site 4 unit 17 and
site 7 unit 21), whose few steps back are under 0.1 ms and which also repeat spike times; no
measure here depends on the order. They run from about 0 to 622 s, and 96
to 98% of the spikes fall within the stimulus. `position` is a pair: a lateral offset of -50, 0
or 50 um (the three shanks' columns of the a1x32-poly3) and a depth of 153 to 662 um below the
surface, 150 um above the tip at the deepest. Sites 13 and 14 have one first trigger at 1.02 s
where the others have 1.00 s. Site pairs (1, 2), (3, 4) and so on share a tip depth, so they may
be one penetration recorded with both stimuli; the next cell tests whether any unit appears in
two files.

In [ ]:
# Units deposited twice: identical spike trains, or the same mean waveform on the same probe channel.
dups, best = [], []
for a, b in itertools.combinations(range(len(cne.SITES)), 2):
    A, B = cne.SITES[a], cne.SITES[b]
    for ua in A["units"]:
        for ub in B["units"]:
            if len(ua["spikes_ms"]) == len(ub["spikes_ms"]) and np.allclose(ua["spikes_ms"], ub["spikes_ms"]):
                dups.append((A["site"], B["site"]))
    if A["site"] % 2 == 1 and B["site"] == A["site"] + 1:
        rs = []
        for ua in A["units"]:
            cands = [np.corrcoef(ua["waveform"][:, -1], ub["waveform"][:, -1])[0, 1]
                     for ub in B["units"] if ub["probechan"] == ua["probechan"]]
            rs.append(max(cands) if cands else np.nan)
        best.append({"sites": f"{A['site']}-{B['site']}", "units_with_same_channel": int(np.isfinite(rs).sum()),
                     "median_best_waveform_r": round(float(np.nanmedian(rs)), 3),
                     "r_above_0.99": int(np.nansum(np.array(rs) > 0.99))})
within = sum(1 for s in cne.SITES for ua, ub in itertools.combinations(s["units"], 2)
             if len(ua["spikes_ms"]) == len(ub["spikes_ms"]) and np.allclose(ua["spikes_ms"], ub["spikes_ms"]))
print("identical spike trains across files:", len(dups), "| within files:", within)
print(pd.DataFrame(best).to_string(index=False))

No spike train appears twice, within or across files. No unit of one file has a mean waveform
correlated above 0.99 with a unit on the same probe channel of the paired file. The paired files
share a tip depth, but their units were sorted separately and cannot be matched, so each file is
counted as one recording, as the paper counts 16 penetrations.

In [ ]:
# Clock check: the STRF of each unit with its spikes on the stimulus clock against the same spikes
# shifted by 30 s (circularly within the stimulus). A shared clock gives a structured STRF only when
# aligned, with its peak at the 10 to 30 ms latency of AI (Methods, Electrophysiology).
rng_clock = np.random.default_rng(11)
rows = []
for s in cne.SITES:
    stim = cne.STIM[(s["stim"], cne.STRF_BIN_MS)]
    dur = cne.stim_duration_ms(s)
    for k, u in enumerate(s["units"]):
        t = u["stim_ms"][np.isfinite(u["stim_ms"])]
        out = {"site": s["site"], "unit": k}
        for label, tt in (("aligned", t), ("shifted_30s", (t + 30000) % dur)):
            st = cne.sta(stim, cne.segment_index(tt, cne.STRF_BIN_MS, stim.shape[1], cne.STRF_LAGS), cne.STRF_LAGS)
            out[f"ptd_{label}"] = float(st.max() - st.min())
            prof = np.abs(st - st.mean()).max(axis=0)
            out[f"peak_lag_ms_{label}"] = (cne.STRF_LAGS - 1 - int(prof.argmax())) * cne.STRF_BIN_MS + cne.STRF_BIN_MS / 2
        rows.append(out)
clock = pd.DataFrame(rows)
clock["ratio"] = clock.ptd_aligned / clock.ptd_shifted_30s
summary = clock.groupby("site").agg(units=("unit", "size"), median_ptd_ratio=("ratio", "median"),
                                    frac_ratio_above_1_5=("ratio", lambda r: (r > 1.5).mean()),
                                    median_peak_lag_aligned=("peak_lag_ms_aligned", "median"),
                                    median_peak_lag_shifted=("peak_lag_ms_shifted_30s", "median"))
print(summary.round(2).to_string())
print("\nall units: median PTD ratio aligned/shifted %.2f; units with ratio > 1.5: %d of %d; peak lags of "
      "units with ratio > 1.5, median %.1f ms (IQR %.1f-%.1f)" % (
          clock.ratio.median(), (clock.ratio > 1.5).sum(), len(clock),
          clock[clock.ratio > 1.5].peak_lag_ms_aligned.median(),
          *clock[clock.ratio > 1.5].peak_lag_ms_aligned.quantile([0.25, 0.75])))

In all 16 recordings the STRFs are stronger with the spikes on the stimulus clock than with the
same spikes shifted by 30 s (median PTD ratio 1.03 to 2.79 per recording, 1.40 over 655 units).
In the aligned STRFs the peak falls at a median lag of 12.5 to 32.5 ms per recording, within or
near the 10 to 30 ms latency of AI. In the shifted STRFs it falls at 42.5 to 62.5 ms, the middle
of the window, where a peak with no structure lands. For the 289 units with a ratio above 1.5 the
median peak lag is 12.5 ms. The triggers and spike times therefore share a clock, and the mapping
of 667 stimulus columns to each trigger interval is consistent with the data. The check is weak
in sites 9, 15 and 13 (median ratios 1.03 to 1.13; 2 to 17% of units above 1.5), where few units
have strong STRFs.

## cNE detection on the 16 recordings

The detection of the claims file's opening comment, at 10 ms, with 100 circular shuffles for the
membership and activity thresholds. Membership uses IC sign reading (a) unless a claim says
otherwise; reading (b) is shown where the claims file names it. Units without spikes inside the
stimulus would be left out; the count is printed.

In [ ]:
t0 = time.time()
main = dict(zip(SITE_IDS, pmap(cne.run_main, IDX, [1000 + s for s in SITE_IDS])))
print(f"detection on {len(main)} recordings in {time.time() - t0:.0f} s")
print("units left out for having no spikes:", sum(d["n_units"] - len(d["kept"]) for d in main.values()))
rows = []
for s in cne.SITES:
    d = main[s["site"]]
    n = len(s["units"])
    rows.append({"site": s["site"], "stim": s["stim"], "units": n, "cnes": d["n"],
                 "mp_bound": round(d["bound"], 3),
                 "member_threshold_a": tuple(np.round(d["thresholds"]["a"], 3)),
                 "member_threshold_b": tuple(np.round(d["thresholds"]["b"], 3)),
                 "1.5/sqrt(n)": round(1.5 / np.sqrt(n), 3),
                 "sizes_a": [len(m) for m in cne.member_sets(d, "a")],
                 "sizes_b": [len(m) for m in cne.member_sets(d, "b")]})
per_site = pd.DataFrame(rows)
print(per_site.to_string(index=False))
print("\nfastICA iterations on the real data (limit 1000; 0 where one cNE needs no ICA):",
      {s: main[s].get("ica_iter") for s in SITE_IDS})
sh = np.concatenate([main[s]["shuffle_ica_iter"] for s in SITE_IDS])
print(f"shuffles: {len(sh)} ICA runs, {int((sh >= cne.SHUFFLE_ICA_ITER).sum())} stopped at the limit of "
      f"{cne.SHUFFLE_ICA_ITER} iterations (median {np.median(sh):.0f})")

The membership thresholds are close to plus or minus 1.5/sqrt(n), the spread of the weights of a
random unit vector over n units: the shuffled ICs carry no structure, as the Methods expect.
fastICA converged on the real data of every recording within 23 to 124 iterations (limit 1000),
and on all 1600 shuffles within the limit of 200.

## fig7-totals

> Across 16 penetrations, 104 cNEs were identified from 655 isolated neurons, an average of
> 6.5 +/- 2.0 cNEs per penetration.

In [ ]:
print("penetrations:", len(per_site), "| units:", per_site.units.sum())
print("cNEs in total:", per_site.cnes.sum())
print("cNEs per penetration: mean %.2f, SD %.2f (n-1), SD %.2f (n); range %d to %d" % (
    per_site.cnes.mean(), per_site.cnes.std(), per_site.cnes.std(ddof=0), per_site.cnes.min(), per_site.cnes.max()))
print(per_site.groupby("stim").cnes.agg(["count", "sum", "mean"]).to_string())

**Result: reproduced.** The data set holds 16 recordings and 655 units, the paper's counts. The
detection finds 104 cNEs, 6.50 +/- 1.97 per recording (range 3 to 10), against the paper's 104 and
6.5 +/- 2.0.

## fig4-example-nine-cnes

> In an example columnar recording with 38 isolated neurons, the cNE detection found nine cNEs,
> with 2 to 6 member neurons each.

In [ ]:
ex = per_site[per_site.units == 38]
print("recordings with 38 units:", ex.site.tolist())
for site in ex.site:
    d = main[site]
    for r in cne.SIGN_READINGS:
        sizes = [len(m) for m in cne.member_sets(d, r)]
        pos = [len(m["pos"]) for m in d["members"][r]]
        print(f"site {site}, sign reading ({r}): {d['n']} cNEs; sizes {sorted(sizes)} (range {min(sizes)}-{max(sizes)}); "
              f"positive members only {sorted(pos)}")
# Seed dependence of the example: the detection with five other seeds
for k in range(5):
    d = cne.detect(cne.binned(cne.SITES[SITE_IDS.index(ex.site.iloc[0])], cne.BIN_MS), 50 + k, activity_null=False)
    print(f"seed {50 + k}: {d['n']} cNEs, sizes (a) {sorted(len(m) for m in cne.member_sets(d, 'a'))}")
d = main[ex.site.iloc[0]]
fig, ax = plt.subplots(figsize=(7, 3))
ax.imshow(cne.orient(d["w"], "a"), aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("cNE"); ax.set_ylabel("unit"); ax.set_title(f"IC weights, site {ex.site.iloc[0]} (all within -1 to 1)")
plt.show()

**Result: reproduced.** Site 10 is the only recording with 38 units. The detection finds 9 cNEs
with 2 to 6 members under sign reading (a), the paper's 9 and 2 to 6, and the same with five
other seeds. Under reading (b) the sizes are 2 to 5, within the tolerance of one.

## fig7a-number-vs-neurons

> The number of cNEs per penetration increased with the number of isolated neurons (R2 = 0.74),
> and was about 15% of the number of recorded neurons.

In [ ]:
fit = stats.linregress(per_site.units, per_site.cnes)
print("least-squares line of cNEs on units: slope %.3f, intercept %.2f, R2 %.3f, F-test p %.2g (n = %d)" % (
    fit.slope, fit.intercept, fit.rvalue ** 2, fit.pvalue, len(per_site)))
print("fraction (a) mean of cNEs/units: %.1f%%" % (100 * (per_site.cnes / per_site.units).mean()))
print("fraction (b) slope: %.1f%%" % (100 * fit.slope))
print("fraction (c) total cNEs / total units: %.1f%%" % (100 * per_site.cnes.sum() / per_site.units.sum()))
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(per_site.units, per_site.cnes, "o")
ax.set_xlabel("units"); ax.set_ylabel("cNEs"); plt.show()

**Result: reproduced.** The number of cNEs rises with the number of units: slope 0.140, R2 0.745,
F-test p = 1.7e-05 (paper R2 0.74, p = 1.68e-5). The fraction is 16.1% (a), 14.0% (b) or 15.9% (c),
all within 15 +/- 1.5%.

## fig7b-size-vs-neurons

> The mean cNE size was 7.4 +/- 2.5 neurons, about 17% of the recorded neurons, and increased
> with the number of isolated neurons, more weakly than the number of cNEs (R2 = 0.33).

In [ ]:
for r in cne.SIGN_READINGS:
    sizes = per_site[f"sizes_{r}"]
    pooled = np.concatenate([np.array(x, float) for x in sizes])
    pooled_frac = np.concatenate([np.array(x, float) / n for x, n in zip(sizes, per_site.units)])
    site_mean = np.array([np.mean(x) for x in sizes])
    fit_b = stats.linregress(per_site.units, site_mean)
    print(f"sign reading ({r})")
    print("  size (a) pooled over %d cNEs: mean %.2f, SD %.2f; (b) mean of per-penetration means %.2f, SD %.2f" % (
        len(pooled), pooled.mean(), pooled.std(ddof=1), site_mean.mean(), site_mean.std(ddof=1)))
    print("  fraction of units: (a) pooled %.1f%%; (b) per penetration %.1f%%" % (
        100 * pooled_frac.mean(), 100 * (site_mean / per_site.units).mean()))
    print("  per-penetration mean size on units: slope %.3f, R2 %.3f, F-test p %.2g" % (
        fit_b.slope, fit_b.rvalue ** 2, fit_b.pvalue))
    print("  sizes range %d to %d" % (pooled.min(), pooled.max()))

**Result: not reproduced.** Mean cNE size rises with the number of units (slope 0.078,
p = 2.3e-06). But the cNEs are smaller than the paper's: 4.58 +/- 1.84 members pooled and 4.34
per recording, against 7.4 +/- 2.5. They are 10.7% of the units, against 17%. The fit is also
tighter, with R2 0.807 against 0.33. Only the positive slope with p < 0.05 holds. Sign reading (b)
gives the same picture (mean 4.73, R2 0.745).

## fig7-negative-weights

> The IC weights of almost all cNE members were positive; 9 cNEs (8.3%) had at least one member
> with a negative weight.

In [ ]:
for r in cne.SIGN_READINGS:
    pos = sum(len(m["pos"]) for d in main.values() for m in d["members"][r])
    neg = sum(len(m["neg"]) for d in main.values() for m in d["members"][r])
    with_neg = sum(1 for d in main.values() for m in d["members"][r] if len(m["neg"]))
    total = sum(d["n"] for d in main.values())
    print(f"sign reading ({r}): cNEs with a negative-weight member {with_neg} of {total} ({100 * with_neg / total:.1f}%); "
          f"member weights positive {pos} of {pos + neg} ({100 * pos / (pos + neg):.1f}%)")

**Result: not reproduced.** Under sign reading (a), 24 of 104 cNEs (23.1%) have a member below
the lower threshold, against the paper's 9. 90.8% of member weights are positive, which passes the
90% reading of "vast majority". Under reading (b), 65 cNEs have one.

## fig7c-membership

> Of the 655 isolated neurons, about 68% belonged to a single cNE, about 9% to none and about 23%
> to more than one.

In [ ]:
for r in cne.SIGN_READINGS:
    counts = []
    for s in cne.SITES:
        sets = cne.member_sets(main[s["site"]], r)
        counts += [sum(1 for m in sets if k in m) for k in range(len(s["units"]))]
    counts = np.array(counts)
    print(f"sign reading ({r}), {len(counts)} units: one cNE {100 * (counts == 1).mean():.1f}%, none "
          f"{100 * (counts == 0).mean():.1f}%, more than one {100 * (counts > 1).mean():.1f}%; most cNEs of one unit: {counts.max()}")

**Result: not reproduced.** Under sign reading (a), 59.2% of the 655 units belong to one cNE,
34.4% to none and 6.4% to more than one. The paper has 68%, 9% and 23%. Reading (b) gives 61.2%,
32.1% and 6.7%. None of the three values is within tolerance under either reading. Fewer units are
members than in the paper, which fits the smaller cNEs of fig7b.

## fig7d-pairs-within-200um

> About 82% of pairs of neurons in the same cNE were recorded on channels less than 200 um
> apart, so cNEs are mostly spatially confined.

In [ ]:
pair_rows = []
for s in cne.SITES:
    for j, m in enumerate(cne.member_sets(main[s["site"]], "a")):
        for a, b in itertools.combinations(m, 2):
            ua, ub = s["units"][a], s["units"][b]
            pair_rows.append({"site": s["site"], "cne": j, "a": a, "b": b,
                              "depth_diff": abs(ua["depth"] - ub["depth"]),
                              "euclid": float(np.hypot(ua["x"] - ub["x"], ua["depth"] - ub["depth"]))})
cpairs = pd.DataFrame(pair_rows)
uniq = cpairs.drop_duplicates(["site", "a", "b"])
for label, df in (("once per cNE", cpairs), ("each pair once", uniq)):
    for dist in ("depth_diff", "euclid"):
        print(f"{label}, {dist}: {100 * (df[dist] < 200).mean():.1f}% of {len(df)} pairs under 200 um "
              f"(range {df[dist].min():.0f} to {df[dist].max():.0f} um)")
print("\nby recording, depth difference, once per cNE:")
print(cpairs.groupby("site").depth_diff.apply(lambda d: round(100 * (d < 200).mean(), 1)).to_string())

**Result: reproduced.** 88.5% of the 1025 member pairs (88.3% of 1012 unique pairs) are less
than 200 um apart, by depth difference and by Euclidean distance alike. This is within 82 +/- 8.2%.
Per recording it ranges from 61.4% to 100%.

## fig7e-no-depth-bias

> cNEs and their member neurons were found across the whole explored depth with no bias to any
> depth; the depths of cNE members and of all recorded neurons were similarly distributed.

In [ ]:
all_depth, mem_depth, cne_depth = [], [], []
for s in cne.SITES:
    sets = cne.member_sets(main[s["site"]], "a")
    inany = set().union(*[set(m.tolist()) for m in sets]) if sets else set()
    all_depth += [u["depth"] for u in s["units"]]
    mem_depth += [s["units"][k]["depth"] for k in sorted(inany)]
    cne_depth += [np.median([s["units"][k]["depth"] for k in m]) for m in sets if len(m)]
all_depth, mem_depth, cne_depth = map(np.array, (all_depth, mem_depth, cne_depth))
ks = stats.ks_2samp(mem_depth, all_depth)
mw = stats.mannwhitneyu(mem_depth, all_depth)
print(f"members {len(mem_depth)}, all units {len(all_depth)}, cNEs {len(cne_depth)}")
print("median depth: members %.0f, all %.0f, difference %.0f um; cNE depth range %.0f to %.0f um (units %.0f to %.0f)" % (
    np.median(mem_depth), np.median(all_depth), np.median(mem_depth) - np.median(all_depth),
    cne_depth.min(), cne_depth.max(), all_depth.min(), all_depth.max()))
print("(a) two-sample KS: D = %.3f, p = %.3g" % (ks.statistic, ks.pvalue))
print("(b) Mann-Whitney U: U = %.0f, p = %.3g" % (mw.statistic, mw.pvalue))

non_depth = []
for s in cne.SITES:
    sets = cne.member_sets(main[s["site"]], "a")
    inany = set().union(*[set(m.tolist()) for m in sets]) if sets else set()
    non_depth += [u["depth"] for k, u in enumerate(s["units"]) if k not in inany]
print("variant, members against non-members (%d): KS p = %.3g, MWU p = %.3g" % (
    len(non_depth), stats.ks_2samp(mem_depth, non_depth).pvalue, stats.mannwhitneyu(mem_depth, non_depth).pvalue))

**Result: reproduced (resting on the choice of test).** The 430 members and all 655 units have
median depths of 460 and 450 um. Neither test finds a difference: KS p = 0.999, Mann-Whitney
p = 0.474. Members against non-members also show none (KS p = 0.498, MWU p = 0.116). cNE depths
run from 200 to 662 um, against 150 to 662 um for the units. The paper names no test, so the
outcome rests on these.

## fig7f-span-200um

> Most cNEs (about 70%) spanned 200 um of cortical depth or less, though a few spanned most of
> the recorded column.

In [ ]:
spans = []
for s in cne.SITES:
    for m in cne.member_sets(main[s["site"]], "a"):
        d = [s["units"][k]["depth"] for k in m]
        spans.append(max(d) - min(d))
spans = np.array(spans)
print(f"{len(spans)} cNEs: span <= 200 um {100 * (spans <= 200).mean():.1f}%; median {np.median(spans):.0f} um; "
      f"max {spans.max():.0f} um; over 400 um: {(spans > 400).sum()}")
for r in ("b",):
    sp_b = np.array([max(s["units"][k]["depth"] for k in m) - min(s["units"][k]["depth"] for k in m)
                     for s in cne.SITES for m in cne.member_sets(main[s["site"]], r) if len(m)])
    print(f"sign reading ({r}): span <= 200 um {100 * (sp_b <= 200).mean():.1f}% of {len(sp_b)}")

**Result: not reproduced.** 80.8% of the 104 cNEs span 200 um or less (78.8% under sign reading
(b)), above the 63 to 77% the tolerance allows. Two cNEs span more than 400 um, up to 500 um. The
cNEs are more confined than the paper's, as expected of smaller cNEs.

## Pairwise correlation functions

For every pair of units of a recording: the PWC (1-ms lag bins, +-100 ms and its central
+-50 ms), its sharpness and absolute peak delay, the correlation coefficient of the 10-ms binned
trains, the depth difference and Euclidean distance of the channels, and whether the pair shares
a cNE (sign reading a). The PWCs use the spikes inside the stimulus period, on the stimulus clock,
as the correlation coefficient does. As variants the claims file does not name, the sharpness is
also given with a baseline subtracted (the count per bin expected from the two rates, or the mean
of the outer +-50 to 100 ms of the histogram, the rest clipped at zero) and over +-10 and +-20 ms.

In [ ]:
t0 = time.time()
pairs = pd.DataFrame([r for rows in pmap(cne.run_pairs, IDX, [main[s] for s in SITE_IDS]) for r in rows])
print(f"{len(pairs)} pairs in {time.time() - t0:.0f} s; pairs without any spike difference within +-50 ms: "
      f"{(pairs.count_50 == 0).sum()}")
print(pairs[["r", "sharp_50", "delay_50", "sharp_100", "delay_100", "depth_diff", "euclid"]].describe().round(3).to_string())
print("\nsharpness variants (not readings of the claims file): baseline subtracted, from the two rates (ind) or the "
      "outer +-50 to 100 ms (flank), and narrower windows")
print(pairs[["sharp_50_ind", "sharp_50_flank", "sharp_100_ind", "sharp_100_flank", "sharp_20", "sharp_10"]]
      .describe().round(3).to_string())

## fig2bd-pwc-measures-covary

> Across all penetrations, the sharpness and peak delay of pairwise correlation (PWC) functions
> covaried with the pairs' spike train correlation coefficients, accounting for about 50% of the
> variance.

In [ ]:
def r2(x, y, log=False):
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    if log:
        ok &= (x > 0) & (y > 0)
        x, y = np.log10(x), np.log10(y)
    f = stats.linregress(x[ok], y[ok])
    return f.rvalue ** 2, f.slope, int(ok.sum())


rows = []
for lag in (50, 100):
    for log in (False, True):
        rs, ss, ns = r2(pairs.r, pairs[f"sharp_{lag}"], log)
        rd, sd, nd = r2(pairs.r, pairs[f"delay_{lag}"], log)
        rows.append({"lags": f"+-{lag} ms", "axes": "log" if log else "linear", "R2_sharpness": round(rs, 3),
                     "slope_sharpness": round(ss, 3), "pairs_sharpness": ns, "R2_delay": round(rd, 3),
                     "slope_delay": round(sd, 3), "pairs_delay": nd})
print(pd.DataFrame(rows).to_string(index=False))
print("\nR2 of sharpness against delay, +-50 ms, linear: %.3f" % r2(pairs.sharp_50, pairs.delay_50)[0])
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(pairs.r, pairs.sharp_50, ".", ms=1); axs[0].set_xlabel("r (10 ms)"); axs[0].set_ylabel("sharpness (ms), +-50")
axs[1].plot(pairs.r, pairs.delay_50, ".", ms=1); axs[1].set_xlabel("r (10 ms)"); axs[1].set_ylabel("|peak delay| (ms), +-50")
plt.tight_layout(); plt.show()
print("plotted: r %.3f to %.3f; sharpness %.0f to %.0f ms; delay %.0f to %.0f ms (all points within the axes)" % (
    pairs.r.min(), pairs.r.max(), pairs.sharp_50.min(), pairs.sharp_50.max(), pairs.delay_50.min(), pairs.delay_50.max()))
rows = []
for col in ("sharp_50_ind", "sharp_50_flank", "sharp_100_ind", "sharp_100_flank", "sharp_20", "sharp_10"):
    for log in (False, True):
        rs, ss, ns = r2(pairs.r, pairs[col], log)
        rows.append({"sharpness variant": col, "axes": "log" if log else "linear", "R2_sharpness": round(rs, 3),
                     "slope": round(ss, 3), "pairs": ns})
print("\nvariants of the sharpness the claims file does not name (the peak delay is the same as above):")
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** Sharpness and peak delay both decrease with the correlation
coefficient. The R2 values are 0.13 and 0.11 (+-50 ms, linear), 0.16 and 0.18 (log), 0.18 and 0.10
(+-100 ms, linear) and 0.23 and 0.25 (log). All are well below 0.5 +/- 0.05. The sharpness
variants the claims file does not name (a baseline subtracted, or windows of +-10 and +-20 ms) give
R2 of 0.045 to 0.315 on linear axes and 0.016 to 0.551 on log axes; only one, the +-100 ms PWC with
the mean of its outer lags subtracted, on log axes, comes near 0.5 (0.551, just outside), and the
delay R2 under the same lags is 0.25. No reading or variant puts both R2 values in tolerance.

## fig2eg-distance-weak

> The correlation between the pairwise synchrony measures (sharpness, peak delay, correlation
> coefficient) and the distance between the pair's channels explained only about 10 to 30% of
> the variance.

In [ ]:
rows = []
for dist in ("depth_diff", "euclid"):
    for lag in (50, 100):
        vals = {"sharpness": r2(pairs[dist], pairs[f"sharp_{lag}"]), "delay": r2(pairs[dist], pairs[f"delay_{lag}"]),
                "r": r2(pairs[dist], pairs.r)}
        rows.append({"distance": dist, "lags": f"+-{lag} ms", **{f"R2_{k}": round(v[0], 3) for k, v in vals.items()},
                     "in_0.09_0.33": sum(0.09 <= v[0] <= 0.33 for v in vals.values())})
print(pd.DataFrame(rows).to_string(index=False))
print("\nvariant, log axes for the measures (pairs with positive values), depth difference, +-50 ms:",
      {k: round(r2(pairs.depth_diff, pairs[c], True)[0], 3) for k, c in (("sharpness", "sharp_50"), ("delay", "delay_50"))})

**Result: not reproduced.** Distance explains even less of the variance than the paper says.
The R2 values are 0.005 to 0.055 for sharpness and delay and 0.032 to 0.037 for the correlation
coefficient, under both distances and both lag ranges. All fall below the 0.09 to 0.33 interval.
The statement's "only" holds, but its range does not.

## fig6j-sharpness-cne-pairs

> In every penetration, spike train PWCs were significantly sharper (narrower) for pairs of
> neurons in the same cNE than for pairs not in the same cNE.

In [ ]:
pairs["cne_pair"] = pairs.n_shared > 0
rows = []
for lag in (50, 100):
    for site, g in pairs.groupby("site"):
        a = g[g.cne_pair][f"sharp_{lag}"].dropna()
        b = g[~g.cne_pair][f"sharp_{lag}"].dropna()
        p = stats.mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan
        rows.append({"lags": lag, "site": site, "cne_pairs": len(a), "non_cne_pairs": len(b),
                     "median_cne": a.median(), "median_non": b.median(), "p": p,
                     "sharper_p<0.001": bool(p < 0.001 and a.median() < b.median())})
sj = pd.DataFrame(rows)
print(sj.round(4).to_string(index=False))
print("\npenetrations with cNE pairs sharper at p < 0.001:", sj.groupby("lags")["sharper_p<0.001"].sum().to_dict())
print("cNE / non-cNE pairs in total:", int(pairs.cne_pair.sum()), int((~pairs.cne_pair).sum()),
      "(paper, Supplementary file 1: 3087 / 8838)")

# Variants: non-cNE pairs restricted to pairs of two cNE members that share no cNE (the verifier's reading of
# the paper's pair counts), and the sharpness variants of the pair table.
rows = []
for col in ("sharp_50", "sharp_100", "sharp_50_ind", "sharp_50_flank", "sharp_100_ind", "sharp_100_flank"):
    for restrict in (False, True):
        n, pv, tot = 0, [], [0, 0]
        for site, g in pairs.groupby("site"):
            a = g[g.cne_pair][col].dropna()
            b = g[~g.cne_pair & (g.both_members if restrict else True)][col].dropna()
            p = stats.mannwhitneyu(a, b).pvalue
            n += bool(p < 0.001 and a.median() < b.median())
            pv.append(p)
            tot = [tot[0] + len(a), tot[1] + len(b)]
        rows.append({"sharpness": col, "non-cNE pairs": "both members" if restrict else "all",
                     "cNE pairs": tot[0], "non-cNE": tot[1], "sites sharper at p<0.001": n,
                     "largest p": float(np.max(pv))})
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** With +-50 ms lags, cNE pairs have sharper PWCs at p < 0.001 in
14 of 16 recordings. The two exceptions have few cNE pairs: site 5 (5 pairs, p = 0.023) and site 6
(10 pairs, p = 0.074); cNE pairs still have the lower median sharpness in both. With +-100 ms it
holds in 12. There are 1012 cNE pairs and 13162 non-cNE pairs, against the paper's 3087 and 8838.
With the non-cNE pairs restricted to pairs of two cNE members that share no cNE (5728 pairs), a
reading of the paper's counts asked of a reader, the result is the same, 14 (+-50 ms) and 12
(+-100 ms). With a baseline subtracted from the PWCs (variants the claims file does not name), 11
to 14 recordings pass. No variant reaches 16.

## fig6-member-pwc-sharpness-delay

> PWCs between members of the same cNE are sharp, with a median sharpness of about 2 ms, a mean
> sharpness of 3.39 +/- 1.21 ms and a mean absolute peak delay of 0.76 +/- 1.74 ms.

In [ ]:
cp = pairs[pairs.cne_pair]
rows = []
for lag in (50, 100):
    s_, d_ = cp[f"sharp_{lag}"].dropna(), cp[f"delay_{lag}"].dropna()
    rows.append({"lags": lag, "reading": "(a) pooled", "median_sharp": s_.median(), "mean_sharp": s_.mean(),
                 "sd_sharp": s_.std(), "mean_abs_delay": d_.mean(), "sd_delay": d_.std(), "pairs": len(s_)})
    g = cp.groupby("site")
    ms, md, mm = g[f"sharp_{lag}"].mean(), g[f"delay_{lag}"].mean(), g[f"sharp_{lag}"].median()
    rows.append({"lags": lag, "reading": "(b) per penetration", "median_sharp": mm.mean(), "mean_sharp": ms.mean(),
                 "sd_sharp": ms.std(), "mean_abs_delay": md.mean(), "sd_delay": md.std(), "pairs": len(ms)})
print(pd.DataFrame(rows).round(2).to_string(index=False))
print("\nfraction of cNE pairs with peak delay 0 or 1 ms (+-50): %.2f" % (cp.delay_50 <= 1).mean())
print("non-cNE pairs, +-50 ms: median sharpness %.1f ms, mean %.1f ms" % (
    pairs[~pairs.cne_pair].sharp_50.median(), pairs[~pairs.cne_pair].sharp_50.mean()))
rows = []
for col in ("sharp_50_ind", "sharp_50_flank", "sharp_100_ind", "sharp_100_flank", "sharp_20", "sharp_10"):
    s_ = cp[col].dropna()
    rows.append({"sharpness variant": col, "median": s_.median(), "mean": round(s_.mean(), 2),
                 "sd": round(s_.std(), 2), "pairs": len(s_),
                 "non-cNE median": pairs[~pairs.cne_pair][col].median()})
print("\nvariants the claims file does not name, cNE pairs pooled:")
print(pd.DataFrame(rows).to_string(index=False))
flat = {lag: np.ones(2 * lag + 1) + np.eye(1, 2 * lag + 1, lag).ravel() * 1e-9 for lag in (10, 50)}
print("half-count width of a flat histogram with its peak at 0: +-10 ms %.0f ms, +-50 ms %.0f ms" % (
    cne.pwc_measures(flat[10])[0], cne.pwc_measures(flat[50])[0]))

**Result: not reproduced.** For the 1012 cNE pairs (+-50 ms) the median sharpness is 12.0 ms and
the mean 12.52 +/- 5.88 ms, against about 2 ms and 3.39 +/- 1.21 ms. The mean absolute peak delay of
1.82 ms is within 0.76 +/- 1.74 ms. Per recording the values are 12.2, 12.5 and 1.54 ms. With
+-100 ms the sharpness is larger still (33 ms). In 80% of cNE pairs the PWC peaks at 0 or 1 ms. The
half-count width includes the flat base of the histogram: a flat histogram gives 25 ms over
+-50 ms. With a baseline subtracted (variants the claims file does not name) the cNE-pair median is
6 to 16 ms, and over +-20 ms it is 6 ms. Only a +-10 ms window brings it to 3 ms (mean 3.11), where
a flat histogram already gives 5 ms, so that the window sets the value; even there the median is
outside 2 +/- 0.2 ms. The paper does not give the lag range or any baseline handling (a question
for its authors).

## fig6c-spike-pwc-narrower-than-strf-pwc

> For pairs of neurons in the same cNE, spike train PWCs were much narrower than the PWCs of their
> STRFs' temporal profiles, while for pairs not in the same cNE the spike train PWCs were often as
> wide as the STRF PWCs.

STRFs at 1 ms with 50 lags (as in the data set's sample code, 50 ms); the temporal profile is the
mean over frequency at each lag (Figure 6B, E legend); the STRF PWC is the cross-correlation of
the two profiles (lags -49 to 49 ms), its sharpness measured as for spike PWCs. A count
histogram cannot be negative and a cross-correlation can, so the STRF PWC is made non-negative
in one of two ways, both run: (i) negative values set to zero, (ii) its minimum subtracted. The
spike PWCs use the +-50 ms reading.

In [ ]:
t0 = time.time()
strf1 = dict(zip(SITE_IDS, pmap(cne.run_strf_profiles, IDX)))
prof = {s: [st.mean(axis=0) - st.mean() for st in v] for s, v in strf1.items()}
overlap = np.correlate(np.ones(50), np.ones(50), mode="full")
strf_readings = {"(i) negative values set to zero": lambda x: np.clip(x, 0, None),
                 "(ii) minimum subtracted": lambda x: x - x.min(),
                 "(iii) absolute value": np.abs,
                 "(iv) divided by the overlap, minimum subtracted": lambda x: x / overlap - (x / overlap).min()}
cols = {}
for k, (name, f) in enumerate(strf_readings.items()):
    vals = []
    for row in pairs.itertuples():
        xc = np.correlate(prof[row.site][row.a], prof[row.site][row.b], mode="full")
        vals.append(cne.pwc_measures(f(xc))[0])
    cols[name] = f"strf_sharp_{k}"
    pairs[f"strf_sharp_{k}"] = vals
print(f"STRFs and STRF PWCs in {time.time() - t0:.0f} s; pairs without a positive STRF PWC: "
      f"{sum(pairs[c].isna().sum() for c in cols.values())}")
rows = []
for name, col in cols.items():
    for spike in ("sharp_50", "sharp_50_ind", "sharp_50_flank"):
        ratio = pairs[spike] / pairs[col]
        a, b = ratio[pairs.cne_pair].dropna(), ratio[~pairs.cne_pair].dropna()
        cpp, npp = pairs[pairs.cne_pair], pairs[~pairs.cne_pair]
        rows.append({"STRF PWC": name, "spike PWC": spike, "STRF median (ms)": cpp[col].median(),
                     "spike median (ms)": cpp[spike].median(),
                     "% cNE pairs spike sharper": round(100 * (cpp[spike] < cpp[col]).mean(), 1),
                     "% non-cNE spike sharper": round(100 * (npp[spike] < npp[col]).mean(), 1),
                     "ratio cNE": round(a.median(), 3), "ratio non": round(b.median(), 3),
                     "MWU p": stats.mannwhitneyu(a, b).pvalue})
print(pd.DataFrame(rows).to_string(index=False))
by = pairs.assign(sharper=pairs.sharp_50 < pairs[cols["(ii) minimum subtracted"]]).groupby(["site", "cne_pair"]).sharper.mean().unstack()
print("reading (ii), raw +-50 ms spike PWC: fraction with spike PWC sharper, by recording (columns: cNE pair):\n",
      by.round(2).T.to_string())

**Result: reproduced, resting on an assumption.** The outcome depends on how the cross-correlation
of the two temporal profiles, which has negative values, is made non-negative before its sharpness
is measured. Setting the negative values to zero (i) leaves only the central lobe: the STRF PWCs
then have a median sharpness of 4.0 ms and are no wider than the spike PWCs in 97.4% of non-cNE
pairs, and
spike PWCs are sharper in only 6.9% of cNE pairs (27.7% and 34.3% with a baseline subtracted from
the spike PWCs). Under the three other readings, minimum subtracted (ii), absolute value (iii), and
divided by the number of overlapping lags with its minimum subtracted (iv, which removes the taper
of a finite cross-correlation), the STRF PWCs have medians of 13 to 17 ms. The spike PWC is then
sharper in 58.4 to 76.2% of cNE pairs with the raw +-50 ms PWC and in 87.9 to 95.8% with a baseline
subtracted, and in every combination the spike/STRF ratio is lower for cNE pairs (Mann-Whitney p =
2.0e-76 to 1.1e-39 for the raw PWC). The ratio is also lower for cNE pairs under (i) (p = 3.3e-61).
Reading (ii) is adopted, with (iii) and (iv) agreeing: the Results call "PWCs between the temporal
profiles of STRFs" "broad" and say that non-cNE spike PWCs are "often as wide as" them, which (i)
contradicts.

## fig4s2-interleaved-halves-stable

> cNEs detected independently in two interleaved halves of a recording were the same, with about
> 96% of the ICs of one half having a significant match in the other.

In [ ]:
t0 = time.time()
halves = dict(zip(SITE_IDS, pmap(cne.run_halves, IDX, [2000 + s for s in SITE_IDS])))
print(f"halves in {time.time() - t0:.0f} s")


def full(w, kept, n):
    out = np.zeros((n, w.shape[1]))
    out[kept] = w
    return out


def abscorr(x, y):
    if x.shape[1] == 0 or y.shape[1] == 0:
        return np.zeros((x.shape[1], y.shape[1]))
    return np.abs(np.corrcoef(x.T, y.T)[:x.shape[1], x.shape[1]:])


rows = []
for s in cne.SITES:
    h, n = halves[s["site"]], len(s["units"])
    A, B = full(h["A"]["w"], h["A"]["kept"], n), full(h["B"]["w"], h["B"]["kept"], n)
    sA = [full(w, h["A"]["kept"], n) for w in h["A"]["shuffled_w"]]
    sB = [full(w, h["B"]["kept"], n) for w in h["B"]["shuffled_w"]]
    real = abscorr(A, B)
    null_a = np.concatenate([abscorr(x, y).ravel() for x, y in zip(sA, sB)])
    null_b = np.concatenate([abscorr(A, y).ravel() for y in sB] + [abscorr(B, x).ravel() for x in sA])
    for reading, null in (("a", null_a), ("b", null_b)):
        thr = np.percentile(null, 99) if len(null) else np.nan
        mA = (real.max(axis=1) > thr) if real.size else np.zeros(A.shape[1], bool)
        mB = (real.max(axis=0) > thr) if real.size else np.zeros(B.shape[1], bool)
        rows.append({"site": s["site"], "reading": reading, "ICs_A": A.shape[1], "ICs_B": B.shape[1],
                     "threshold": thr, "matched_A": int(mA.sum()), "matched_B": int(mB.sum())})
hv = pd.DataFrame(rows)
print(hv.round(3).to_string(index=False))
for reading, g in hv.groupby("reading"):
    tot = g.ICs_A.sum() + g.ICs_B.sum()
    print(f"reading ({reading}): {g.matched_A.sum() + g.matched_B.sum()} of {tot} ICs matched "
          f"({100 * (g.matched_A.sum() + g.matched_B.sum()) / tot:.1f}%); thresholds {g.threshold.min():.3f}-{g.threshold.max():.3f}")

**Result: reproduced.** 192 of 205 ICs of the two interleaved halves (93.7%) have a match above the
99th percentile of the shuffled null, under both null readings (thresholds 0.32 to 0.62). The
paper has about 96%. Per recording the share runs from 81% (site 16, 13 of 16) to 100%.

## fig4s3-train-test-83pct

> cNEs detected in the first half of a recording predicted cNE activity in the second half; of
> 102 training cNEs, about 83% had significant matches to test cNE activity.

In [ ]:
t0 = time.time()
tt = dict(zip(SITE_IDS, pmap(cne.run_traintest, IDX, [3000 + s for s in SITE_IDS])))
print(f"train/test in {time.time() - t0:.0f} s; training cNEs {sum(v['n_train'] for v in tt.values())}, "
      f"test cNEs {sum(v['n_test'] for v in tt.values())}")
for p in cne.PROJECTIONS:
    matched, null = [], []
    for v in tt.values():
        c = v[p]
        if c.size == 0:
            matched += [-np.inf] * v["n_train"]
            continue
        rmax, cmax = c.argmax(axis=1), c.argmax(axis=0)
        is_max = np.zeros_like(c, bool)
        is_max[np.arange(c.shape[0]), rmax] = True
        is_max[cmax, np.arange(c.shape[1])] = True
        matched += list(c.max(axis=1))
        null += list(c[~is_max])
    thr = np.percentile(null, 95)
    matched = np.array(matched)
    print(f"projection {p}: threshold (95th percentile of {len(null)} non-matched) {thr:.3f}; "
          f"{(matched > thr).sum()} of {len(matched)} training cNEs matched ({100 * (matched > thr).mean():.1f}%); "
          f"matched correlations median {np.median(matched[np.isfinite(matched)]):.3f}")
    # variant: null of values that are not the row maximum only (the Results' wording)
    null2 = []
    for v in tt.values():
        c = v[p]
        if c.size:
            m = np.ones_like(c, bool); m[np.arange(c.shape[0]), c.argmax(axis=1)] = False
            null2 += list(c[m])
    print(f"   variant, null of all non-row-maximum values: threshold {np.percentile(null2, 95):.3f}, "
          f"{100 * (matched > np.percentile(null2, 95)).mean():.1f}% matched")

**Result: not reproduced.** There are 102 training cNEs, the paper's number. But 100 of them
(98.0%, quadratic projection) or 101 (99.0%, linear) match a test cNE above the 95th percentile of
the non-matched correlations, against about 83% (74.7 to 91.3% allowed). With the null taken as
every non-row-maximum value, the figure is 95.1% or 99.0%. The training cNEs predict the test
activity better than the paper reports. The criterion counts a correct count with an out-of-range
percentage as not reproduced.

## Bin sizes (Figure 5)

Detection at 2, 5, 8, 10, 15, 20, 35 and 50 ms, with 100 shuffles each. For Figure 5A, each
cNE's activity is interpolated to 1 ms and given 5000 events, under both projection readings and
both event readings: (a) the 5000 largest 1-ms samples, (b) the 5000 largest local peaks. The
STRF is the event-triggered average (5-ms stimulus bins, 20 lags), and its MI is as in
`cne.strf_mi`. As a check the paper does not describe, the MI is also computed with the STRF
from the events of one half of the stimulus and the projections of the events of the other half
(`cne.strf_mi_heldout`), and the number of distinct detection bins the 5000 events fall in is
counted.

In [ ]:
t0 = time.time()
bins = dict(zip(SITE_IDS, pmap(cne.run_binsizes, IDX, [4000 + s for s in SITE_IDS])))
print(f"bin sizes in {time.time() - t0:.0f} s")
rows = []
for site, v in bins.items():
    for b, r in v.items():
        rows.append({"site": site, "bin": b, "n": r["n"], "mean_size_a": np.mean(r["sizes_a"]) if r["sizes_a"] else np.nan,
                     "mean_size_b": np.mean(r["sizes_b"]) if r["sizes_b"] else np.nan})
bt = pd.DataFrame(rows)
print("check: number of cNEs at 10 ms here and in the main detection agree in",
      sum(bt[(bt.site == s) & (bt.bin == 10)].n.iloc[0] == main[s]["n"] for s in SITE_IDS), f"of {len(SITE_IDS)} recordings")

## fig5a-mi-peaks-10-15ms

> The mutual information between cNE STRFs and the stimulus peaked at bin sizes of about 10 to
> 15 ms.

In [ ]:
rows = []
for b in cne.BIN_SIZES:
    for p in cne.PROJECTIONS:
        for er in ("a", "b"):
            vals = np.concatenate([np.array(v[b]["mi"].get((p, er), [])) for v in bins.values()])
            rows.append({"bin": b, "projection": p, "events": er, "cnes": len(vals),
                         "mean": vals.mean(), "median": vals.median() if hasattr(vals, "median") else np.median(vals)})
mi5 = pd.DataFrame(rows)
for (p, er), g in mi5.groupby(["projection", "events"]):
    print(f"projection {p}, events ({er}):")
    print("   " + g[["bin", "cnes", "mean", "median"]].round(3).to_string(index=False).replace("\n", "\n   "))
    print(f"   peak of the mean at {g.loc[g['mean'].idxmax(), 'bin']} ms, of the median at {g.loc[g['median'].idxmax(), 'bin']} ms")
rows = []
for b in cne.BIN_SIZES:
    for p in cne.PROJECTIONS:
        for er in ("a", "b"):
            ho = np.concatenate([np.array(v[b]["mi_heldout"].get((p, er), [])) for v in bins.values()])
            nd = np.concatenate([np.array(v[b]["distinct_bins"].get((p, er), [])) for v in bins.values()])
            rows.append({"bin": b, "projection": p, "events": er, "cnes": len(ho), "missing": int(np.isnan(ho).sum()),
                         "mean": np.nanmean(ho), "median": np.nanmedian(ho), "median distinct bins": np.median(nd),
                         "min distinct bins": nd.min()})
ho5 = pd.DataFrame(rows)
print("\nMI with the STRF estimated on the events of the other half of the stimulus (a check, not the paper's "
      "method), and the number of distinct detection bins the 5000 events fall in:")
for (p, er), g in ho5.groupby(["projection", "events"]):
    print(f"projection {p}, events ({er}):")
    print("   " + g[["bin", "cnes", "missing", "mean", "median", "median distinct bins", "min distinct bins"]]
          .round(3).to_string(index=False).replace("\n", "\n   "))
    print(f"   peak of the mean at {g.loc[g['mean'].idxmax(), 'bin']} ms, of the median at {g.loc[g['median'].idxmax(), 'bin']} ms")

**Result: not reproduced.** No reading puts the largest MI at 10 or 15 ms. With the 5000 largest
1-ms samples as events (a), MI rises with bin size and peaks at 35 ms under both projections and
both summaries (mean 0.698 linear, 0.730 quadratic). It is nearly flat from 15 ms on (0.669 to 0.698
linear). With the 5000 largest peaks (b), MI is highest at 2 ms and falls with bin size. Under (a)
the 5000 events fall in a median of 230 (linear) or 207 (quadratic) distinct 50-ms bins, against
2857 or 3362 at 2 ms. With the STRF estimated on the other half of the stimulus, the MI is a tenth
to a fifth of the in-sample value (0.059 to 0.082 bits for the means under a) and still does not peak at
10 or 15 ms: under (a) the mean peaks at 5 ms (linear; 0.074 against 0.073 at 8 to 15 ms) or 50 ms
(quadratic), the median at 50 ms; under (b) at 2 ms. The STRF resolution (5 ms, 20 lags) and the
MI bins are this attempt's choices.

## fig5b-number-decreases-with-bin

> The number of detected cNEs decreased as the bin size increased.

In [ ]:
nb_ = bt.groupby("bin").n.agg(["mean", "std", "min", "max"])
print(nb_.round(2).to_string())
rho = stats.spearmanr(nb_.index, nb_["mean"])
print("Spearman rho (mean number against bin size, 8 sizes) %.3f, p = %.3g; 2 ms %.2f against 50 ms %.2f" % (
    rho.correlation, rho.pvalue, nb_["mean"].loc[2], nb_["mean"].loc[50]))
print("recordings with fewer cNEs at 50 than at 2 ms:", int((bt[bt.bin == 50].set_index("site").n < bt[bt.bin == 2].set_index("site").n).sum()), "of 16")

**Result: reproduced.** The mean number of cNEs falls from 11.06 at 2 ms to 3.75 at 50 ms,
monotonically over the eight sizes (Spearman rho = -1.0). Every one of the 16 recordings has fewer
at 50 ms than at 2 ms.

## fig5c-size-increases-with-bin

> The mean number of member neurons per cNE increased as the bin size increased.

In [ ]:
for r in ("a", "b"):
    ms_ = bt.groupby("bin")[f"mean_size_{r}"].agg(["mean", "std", "count"])
    rho = stats.spearmanr(ms_.index, ms_["mean"])
    print(f"sign reading ({r}):\n" + ms_.round(2).to_string())
    print("   Spearman rho %.3f, p = %.3g; 2 ms %.2f against 50 ms %.2f" % (
        rho.correlation, rho.pvalue, ms_["mean"].loc[2], ms_["mean"].loc[50]))

**Result: reproduced.** The mean cNE size rises from 3.88 at 2 ms to 5.11 at 50 ms (Spearman
rho = 0.952, p = 0.00026) under sign reading (a). Under (b) it rises from 3.91 to 5.51
(rho = 1.0). The rise is not strictly monotonic under (a), with small dips at 8 and 20 ms.

## fig10b-ptd-higher and fig10c-strf-mi-higher

Each cNE against each of its members (sign reading a). cNE events are the 10-ms bins of activity
above its 99.9th percentile threshold, at the bin centre: reading (a) every such bin, (b) the
peak bin of each run; under both projection readings. The larger of the event and spike trains
is subsampled to the smaller count. STRFs on 5-ms bins with 20 lags. As a check, the MI is also computed
with the STRF from the other half of the stimulus (`cne.strf_mi_heldout`), and the in-sample test is
repeated with four other subsampling seeds.

In [ ]:
t0 = time.time()
cvm = pd.DataFrame([r for rows in pmap(cne.run_cne_vs_members, IDX, [main[s] for s in SITE_IDS],
                                       [5000 + s for s in SITE_IDS]) for r in rows])
print(f"cNE against member STRFs in {time.time() - t0:.0f} s; pairs per reading:")
print(cvm.groupby(["projection", "events"]).agg(pairs=("count", "size"), median_count=("count", "median"),
                                                min_count=("count", "min")).to_string())
n_members = sum(len(m) for d in main.values() for m in cne.member_sets(d, "a"))
print("cNE and member pairs available:", n_members, "(paper: 703)")

## fig10b-ptd-higher

> cNE STRFs had stronger excitatory and inhibitory subfields than those of their member neurons,
> with a higher peak-trough difference.

In [ ]:
for (p, er), g in cvm.groupby(["projection", "events"]):
    w = stats.wilcoxon(g.ptd_cne, g.ptd_unit)
    print(f"projection {p}, events ({er}): {len(g)} pairs; PTD cNE median {g.ptd_cne.median():.3f}, member "
          f"{g.ptd_unit.median():.3f}; cNE higher in {100 * (g.ptd_cne > g.ptd_unit).mean():.1f}%; signed-rank p = {w.pvalue:.3g}")
    print("   by recording, % cNE higher:", g.groupby("site").apply(lambda x: round(100 * (x.ptd_cne > x.ptd_unit).mean())).to_dict())

**Result: reproduced.** cNE STRFs have a higher PTD than their members' under all four readings.
The signed-rank p is 3.5e-12 (linear, events a), 1.9e-09 (linear, b), 5.0e-04 (quadratic, a) and
1.3e-05 (quadratic, b). The cNE is higher in 54 to 62% of the 476 pairs; the paper has 703 pairs,
as its cNEs are larger.

## fig10c-strf-mi-higher

> cNEs had higher mutual information between their STRFs and single events than member neurons
> had between their STRFs and single spikes.

In [ ]:
for (p, er), g in cvm.groupby(["projection", "events"]):
    w = stats.wilcoxon(g.mi_cne, g.mi_unit)
    print(f"projection {p}, events ({er}): {len(g)} pairs; MI cNE median {g.mi_cne.median():.3f} bits, member "
          f"{g.mi_unit.median():.3f}; cNE higher in {100 * (g.mi_cne > g.mi_unit).mean():.1f}%; signed-rank p = {w.pvalue:.3g}")
    print("   by recording, % cNE higher:", g.groupby("site").apply(lambda x: round(100 * (x.mi_cne > x.mi_unit).mean())).to_dict())
print("\nMI with the STRF from the events or spikes of the other half of the stimulus (a check, not the paper's method):")
for (p, er), g in cvm.groupby(["projection", "events"]):
    h = g.dropna(subset=["mi_cne_heldout", "mi_unit_heldout"])
    w = stats.wilcoxon(h.mi_cne_heldout, h.mi_unit_heldout)
    print(f"projection {p}, events ({er}): {len(h)} pairs ({len(g) - len(h)} without events in a half); MI cNE median "
          f"{h.mi_cne_heldout.median():.3f} bits, member {h.mi_unit_heldout.median():.3f}; cNE higher in "
          f"{100 * (h.mi_cne_heldout > h.mi_unit_heldout).mean():.1f}%; signed-rank p = {w.pvalue:.3g}")

print("\nOther subsampling seeds (in-sample MI as above), signed-rank p:")
t0 = time.time()
rows = []
for seed0 in (15000, 25000, 35000, 45000):
    c_ = pd.DataFrame([r for rr in pmap(cne.run_cne_vs_members, IDX, [main[s] for s in SITE_IDS],
                                        [seed0 + s for s in SITE_IDS], [False] * len(IDX)) for r in rr])
    for (p, er), g in c_.groupby(["projection", "events"]):
        rows.append({"seed": seed0, "reading": f"{p}, {er}", "p": stats.wilcoxon(g.mi_cne, g.mi_unit).pvalue,
                     "cNE higher %": round(100 * (g.mi_cne > g.mi_unit).mean(), 1)})
seeds10 = pd.DataFrame(rows)
print(seeds10.pivot(index="seed", columns="reading", values="p").to_string(float_format=lambda x: f"{x:.2g}"))
print(f"in {time.time() - t0:.0f} s; runs with p < 0.001 and cNE higher, per reading:",
      seeds10.assign(ok=(seeds10.p < 0.001) & (seeds10["cNE higher %"] > 50)).groupby("reading").ok.sum().to_dict())

**Result: reproduced, resting on an assumption.** With the in-sample MI of the criterion, cNE STRF
MI is higher than the members' in 53 to 56% of the 476 pairs, and only the linear projection with
event reading (a) reaches p < 0.001 (p = 6.4e-04); the others give 0.0098 (linear, b), 0.044
(quadratic, a) and 0.15 (quadratic, b). The result depends on the subsampling draw: with four other
seeds the linear (a) reading gives p = 4.2e-06, 0.012, 0.0029 and 4.9e-06, so it passes in 3 of 5
draws, and no other reading passes in any. The MI with the STRF from the other half of the stimulus,
which removes the bias of fitting and testing on the same events, is higher for the cNE under all
four readings (59 to 65% of pairs, p = 2.2e-15 to 6.7e-08). The outcome rests on the linear
projection, one of the claims file's two readings, and the paper's p = 1.6e-59 over 703 pairs is
far stronger than any in-sample test here.

## fig10s1-random-group-strfs

> Multi-unit STRFs of cNE members had higher PTD and higher STRF MI than those of random groups of
> neurons of the same size, with uniqueness indices skewed towards 0.

500 random groups per cNE, as the paper; the UI is the share of the cNE's value and its random
groups' values at or above the cNE's (501 values where 500 groups exist). Two more seeds with 100
groups each show how much the tests move with the draw.

In [ ]:
t0 = time.time()
rg = pd.DataFrame([r for rows in pmap(cne.run_random_groups, IDX, [main[s] for s in SITE_IDS],
                                      [6000 + s for s in SITE_IDS], [500] * len(IDX)) for r in rows])
print(f"random groups in {time.time() - t0:.0f} s; cNEs {len(rg)}, excluded for under 200 spikes {int(rg.excluded.sum())}; "
      f"groups per cNE {rg.n_groups.min()}-{rg.n_groups.max()}; levels used {sorted(rg.level.unique())}")
print(rg[rg.n_groups < 500][["site", "cne", "size", "n_groups", "level", "count"]].to_string(index=False))
ok = rg[~rg.excluded & (rg.n_groups > 0)]
for m in ("ptd", "mi"):
    w = stats.wilcoxon(ok[f"{m}_cne"], ok[f"{m}_rand_median"])
    ks = stats.kstest(ok[f"ui_{m}"], "uniform")
    print(f"{m}: cNE median {ok[f'{m}_cne'].median():.3f}, random median {ok[f'{m}_rand_median'].median():.3f}; "
          f"cNE higher in {100 * (ok[f'{m}_cne'] > ok[f'{m}_rand_median']).mean():.1f}% of {len(ok)}; signed-rank p = {w.pvalue:.3g}; "
          f"UI median {ok[f'ui_{m}'].median():.3f}, KS against uniform D = {ks.statistic:.3f}, p = {ks.pvalue:.3g}")
print("\nOther seeds, 100 random groups each, signed-rank p (PTD, MI) and KS p (PTD, MI):")
t0 = time.time()
for seed0 in (16000, 26000):
    r_ = pd.DataFrame([r for rows in pmap(cne.run_random_groups, IDX, [main[s] for s in SITE_IDS],
                                          [seed0 + s for s in SITE_IDS], [100] * len(IDX)) for r in rows])
    o_ = r_[~r_.excluded & (r_.n_groups > 0)]
    print(f"   seed {seed0}: {len(o_)} cNEs; PTD {stats.wilcoxon(o_.ptd_cne, o_.ptd_rand_median).pvalue:.2g}, "
          f"MI {stats.wilcoxon(o_.mi_cne, o_.mi_rand_median).pvalue:.2g}; KS {stats.kstest(o_.ui_ptd, 'uniform').pvalue:.2g}, "
          f"{stats.kstest(o_.ui_mi, 'uniform').pvalue:.2g}")
print(f"in {time.time() - t0:.0f} s")

**Result: reproduced.** The test covers 103 cNEs, the paper's 103, with 500 random groups each
where the cNE allows; 13 cNEs (twelve with 2 members, one with 3) allow 28 to 228. The summed spikes
of cNE members give a higher PTD than the median random group (3.19 against 2.87; signed-rank p =
3.7e-04) and a higher STRF MI (0.287 against 0.224 bits; p = 3.1e-07). The UIs have medians of 0.353
(PTD) and 0.234 (MI) and differ from uniform (KS p = 9.6e-06 and 1.7e-08). The PTD test is the
weakest part: with two other seeds and 100 groups it gives p = 0.0023 and 0.00061, while the MI and
KS tests stay below 1e-4.

## Models (Figure 11C, D)

200 runs of each model per recording, as the Methods. Shuffled: each 10-ms train shifted
circularly. PR: each spike assigned to a random time bin of the same projection class (15 equal
classes over the full range of the unit's STRF projections), on (a) 10-ms or (b) 5-ms bins, the
latter summed to 10 ms. DG: binary 5-ms trains (counts above one set to one) fitted for means and
pairwise covariances (`cne.fit_dg`, a Python port of the method of CorBinian's
findLatentGaussian01, not the MATLAB code itself), sampled and summed to 10 ms. Membership in
model runs uses the recording's thresholds from the shuffles of the real data, as the shuffled IC
weights follow 1.5/sqrt(n) and do not depend on the data (printed above): an assumption, to fit
the time limit.
The check after the models shows how much the DG fit was changed to make its latent correlation
matrix positive definite, and how well a sampled run matches the real 5-ms rates and covariances.

In [ ]:
t0 = time.time()
models = dict(zip(SITE_IDS, pmap(cne.run_models, IDX, [main[s] for s in SITE_IDS],
                                 [7000 + s for s in SITE_IDS], [200] * len(IDX))))
print(f"models in {time.time() - t0:.0f} s")
rows = []
for s in SITE_IDS:
    real_size = np.mean([len(m) for m in cne.member_sets(main[s], "a")]) if main[s]["n"] else np.nan
    row = {"site": s, "real_n": main[s]["n"], "real_size": real_size}
    for m in cne.MODELS:
        row[f"{m}_n"] = models[s][m]["n"].mean()
        row[f"{m}_size"] = np.nanmean(models[s][m]["size"]) if np.isfinite(models[s][m]["size"]).any() else np.nan
        row[f"{m}_size0"] = np.nan_to_num(models[s][m]["size"], nan=0.0).mean()
        row[f"{m}_runs_without_cne"] = int((models[s][m]["n"] == 0).sum())
    rows.append(row)
mt = pd.DataFrame(rows)
print(mt.round(2).to_string(index=False))

In [ ]:
# Checks of the models: the clipping in the DG fit, and how well one DG sample reproduces the real 5-ms
# binary rates and pairwise covariances; spikes the PR model drops (bins without a full stimulus window).
dgc = pd.DataFrame(pmap(cne.run_dg_check, IDX, [7700 + s for s in SITE_IDS]))
dgc["cov_median_abs_diff / real"] = dgc.cov_median_abs_diff / dgc.cov_real_median_abs
print(dgc[["site", "min_eigenvalue", "max_change", "rate_real_mean", "rate_max_abs_diff", "cov_r", "cov_slope",
           "cov_median_abs_diff / real", "pr_dropped_spikes_10", "pr_dropped_spikes_5", "spikes"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3g}"))

## fig11c-models-number

> Surrogate spike trains from the 'shuffled' and 'preserved receptive field' (PR) models gave
> fewer cNEs than the real data, while the dichotomized Gaussian (DG) model gave more.

In [ ]:
for pr in ("pr10", "pr5"):
    print(f"PR reading {'(a) 10-ms' if pr == 'pr10' else '(b) 5-ms'} bins:")
    for m in ("shuffled", pr, "dg"):
        t = stats.ttest_rel(mt[f"{m}_n"], mt.real_n)
        print(f"   {m}: mean {mt[f'{m}_n'].mean():.2f} against real {mt.real_n.mean():.2f}; t = {t.statistic:.2f}, "
              f"p = {t.pvalue:.3g}, Bonferroni (x3) p = {min(1, 3 * t.pvalue):.3g}; recordings with model fewer: "
              f"{int((mt[f'{m}_n'] < mt.real_n).sum())} of 16")

**Result: reproduced.** Over the 16 recordings (real mean 6.50 cNEs), the shuffled model gives
0.23 cNEs and PR 1.41 (10-ms bins) or 1.52 (5-ms bins), fewer in every recording. DG gives 8.00,
more in 15 of 16. All three Bonferroni-corrected paired t-tests give p < 0.001 (4.6e-09, 3.1e-08 or
4.6e-08, and 1.5e-04), under both PR readings.

## fig11d-models-size

> All three models, including the DG model, gave cNEs with fewer member neurons than the real
> data.

In [ ]:
for zero in (False, True):
    print("runs without cNEs counted as size 0" if zero else "runs without cNEs left out")
    for pr in ("pr10", "pr5"):
        for m in ("shuffled", pr, "dg"):
            col = f"{m}_size0" if zero else f"{m}_size"
            ok = mt[[col, "real_size"]].dropna()
            t = stats.ttest_rel(ok[col], ok.real_size)
            print(f"   {m}: {len(ok)} recordings; mean {ok[col].mean():.2f} against real {ok.real_size.mean():.2f}; "
                  f"t = {t.statistic:.2f}, Bonferroni (x3) p = {min(1, 3 * t.pvalue):.3g}")

**Result: not reproduced.** The model cNEs are not smaller than the real ones (mean 4.34 members).
With runs without cNEs left out, the shuffled model gives 4.95 (larger, corrected p = 0.055), PR
3.80 or 3.84 (p = 0.20 or 0.24) and DG 4.30 (p = 1). With such runs counted as size 0, the shuffled
model is smaller (1.11, p = 1.4e-07), but PR (3.74 or 3.83) and DG are not. At most one of the
three comparisons holds.

## fig11s3b-pr-strf-similarity

> The PR model preserved the neurons' STRFs (median STRF similarity with the real spike trains
> about 0.74), while the shuffled and DG models gave STRF similarities close to zero.

Five runs of each model per recording; the real STRF is from the real train binned as the model
is (10 ms; 5 ms for PR reading b and for DG), spikes at bin centres.

In [ ]:
t0 = time.time()
sim = pmap(cne.run_strf_similarity, IDX, [8000 + s for s in SITE_IDS], [5] * len(IDX))
print(f"STRF similarity in {time.time() - t0:.0f} s")
for m in cne.MODELS:
    v = np.concatenate([x[m] for x in sim])
    print(f"{m}: median {np.median(v):.3f} (IQR {np.percentile(v, 25):.3f} to {np.percentile(v, 75):.3f}) over {len(v)} unit runs")

**Result: reproduced, resting on an assumption.** The median STRF similarity is 0.737 for PR on
5-ms bins and 0.645 for PR on 10-ms bins, against 0.74 +/- 0.074; it is 0.022 for both the shuffled
and DG models, below 0.1. The 10-ms reading falls just outside the tolerance. The 5-ms reading is
adopted, as it is the resolution of the STRFs (an assumption of this attempt) that the PR model
shuffles on, and it is the DG model's bin in the Methods.

## fig11ef-coincidence-ratios

> Normalized to the real data, the coincidence ratios of groups of 2 and 3 cNE members were
> significantly below one for the 'repeat' data and the shuffled, PR and DG models.

Reading (b) of the criterion, the only one the data set allows: groups from the cNEs of the 16
recordings (sign reading a), real and model CRs at 10 ms. 400 unique groups of 2 and of 3 members
of one cNE are drawn from all such groups. Each model is run 200 times per recording, not the
paper's 1000, to fit the time limit (a deviation); the median CR over runs is used. The 'repeat'
CR needs the repeated 10-min RN, which the data set does not hold.

In [ ]:
rng_cr = np.random.default_rng(9000)
groups = {}
for size in (2, 3):
    allg = sorted({(s, g) for s in SITE_IDS for m in cne.member_sets(main[s], "a") for g in itertools.combinations(sorted(m.tolist()), size)})
    pick = rng_cr.choice(len(allg), min(400, len(allg)), replace=False)
    groups[size] = [allg[k] for k in pick]
    print(f"groups of {size}: {len(allg)} available, {len(groups[size])} drawn, from {len({s for s, _ in groups[size]})} recordings")
by_site = {s: [g for size in (2, 3) for (ss, g) in groups[size] if ss == s] for s in SITE_IDS}
t0 = time.time()
cr = pd.DataFrame([r for rows in pmap(cne.run_cr, IDX, [by_site[s] for s in SITE_IDS],
                                      [9000 + s for s in SITE_IDS], [200] * len(IDX)) for r in rows])
cr["key"] = list(zip(cr.site, cr.group))
print(f"CRs in {time.time() - t0:.0f} s")
for size in (2, 3):
    g = cr[cr["size"] == size]
    keep = g.real > 0
    print(f"\ngroups of {size}: {len(g)}; real CR median {g.real.median():.3f}; groups with real CR 0 left out: {(~keep).sum()}")
    g = g[keep]
    for m in cne.MODELS:
        t = stats.ttest_rel(g[m], g.real)
        norm = g[m] / g.real
        print(f"   {m}: normalised CR median {norm.median():.3f}, mean {norm.mean():.3f}, below one in "
              f"{100 * (norm < 1).mean():.1f}%; paired t = {t.statistic:.2f}, p = {t.pvalue:.3g}, "
              f"Bonferroni x4 p = {min(1, 4 * t.pvalue):.3g}")

**Result: partially reproduced.** For the 400 groups of 2, the normalised CRs have medians of
0.159 (shuffled), 0.215 or 0.223 (PR) and 0.776 (DG). For the 372 groups of 3 with a real CR above
zero (28 left out), they are 0.000, 0.000 and 0.730. Every comparison is below one with
Bonferroni p < 1e-35. The 'repeat' comparison cannot be run, as the data set does not hold the
repeated stimulus; by the criterion, the claim is partially reproduced.

## Summary

| Claim | Outcome |
|---|---|
| fig2bd-pwc-measures-covary | not reproduced (R2 0.10 to 0.25, against 0.5) |
| fig2eg-distance-weak | not reproduced (R2 0.005 to 0.055, below 10 to 30%) |
| fig4-example-nine-cnes | reproduced (9 cNEs, 2 to 6 members) |
| fig4s2-interleaved-halves-stable | reproduced (93.7%) |
| fig4s3-train-test-83pct | not reproduced (102 training cNEs, but 98% matched) |
| fig5a-mi-peaks-10-15ms | not reproduced (peak at 35 ms or 2 ms) |
| fig5b-number-decreases-with-bin | reproduced |
| fig5c-size-increases-with-bin | reproduced |
| fig6c-spike-pwc-narrower-than-strf-pwc | reproduced, resting on an assumption |
| fig6j-sharpness-cne-pairs | partially reproduced (14 of 16 recordings) |
| fig6-member-pwc-sharpness-delay | not reproduced (median sharpness 12 ms, against 2) |
| fig7-totals | reproduced (16, 655, 104, 6.5 +/- 2.0) |
| fig7a-number-vs-neurons | reproduced (R2 0.745) |
| fig7b-size-vs-neurons | not reproduced (mean size 4.6, against 7.4) |
| fig7-negative-weights | not reproduced (24 cNEs, against 9) |
| fig7c-membership | not reproduced (59/34/6%, against 68/9/23%) |
| fig7d-pairs-within-200um | reproduced (88.5%) |
| fig7e-no-depth-bias | reproduced, resting on an assumption (the test) |
| fig7f-span-200um | not reproduced (80.8%, against 70%) |
| fig8-spont-evoked-72pct | blocked, data missing |
| fig9c-cne-events-more-mi, fig9s1, fig9s2 | blocked, data missing |
| fig10b-ptd-higher | reproduced |
| fig10c-strf-mi-higher | reproduced, resting on an assumption (linear projection; 3 of 5 draws) |
| fig10s1-random-group-strfs | reproduced |
| fig11ab-real-vs-surrogate | blocked, data missing |
| fig11c-models-number | reproduced |
| fig11d-models-size | not reproduced |
| fig11ef-coincidence-ratios | partially reproduced (models only) |
| fig11s3b-pr-strf-similarity | reproduced, resting on an assumption |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The detection follows the Methods, but the Marcenko-Pastur bound, the fastICA settings, IC sign,
  the activity projection and the event definition are this attempt's readings (claims file and
  `cne.py`). The number of cNEs matches the paper in total (104) and in its dependence on the
  number of units. The cNEs are smaller (4.6 against 7.4 members), so the membership threshold or
  IC scaling probably differs from the authors'. The membership claims (fig7b, fig7c, fig7f,
  fig7-negative-weights) and the pair counts depend on it.
- PWC sharpness, as the half-count width of a raw histogram over +-50 or +-100 ms, includes the
  flat base and cannot give the paper's 2 ms; neither can a baseline subtracted from it (6 to 16
  ms). The paper's lag range or baseline handling is not stated. PWCs use the spikes inside the
  stimulus period.
- STRFs use 5-ms bins and 20 lags (and 1 ms with 50 lags for the temporal profiles), from the
  data set's sample code by the first author; the paper gives no STRF parameters. The MI
  projection bins are a standard choice.
- Fewer iterations than the paper, for time: 200 model runs for the CRs (paper 1000). Membership in
  model runs uses the real data's thresholds. The random groups number 500 per cNE, as the paper.
- The STRF MI is computed on the same events as the STRF, as the Methods describe; with the STRF
  from held-out events (a check) it is a tenth to a fifth as large, and the cNE MI exceeds the
  member MI far more clearly (fig10c).
- The DG model is a Python port of the CorBinian fit (bisection on each pair's latent
  correlation), not the MATLAB code.

What was found about the data set: it holds the 16 columnar recordings and 655 units of
Figures 1 to 7, 10 and 11C to F, with the stimuli, and its counts, units and clocks are
consistent with the descriptor. It does not hold the 5-s repeats (Figure 9), the 15 repeats of
the 10-min RN (Figure 11A, B, and the 'repeat' CRs) or the 64-channel recordings (Figure 8).
Paired files (sites 1 and 2, 3 and 4, and so on) share a tip depth and may be one penetration
recorded with both stimuli; the descriptor does not say, and no unit can be matched across them.
Two units have spike times out of order by under 0.1 ms.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min; peak memory of the main process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")